In [8]:
import csv
import time
import threading
import queue
import ctypes
import os
import math

from neurapy.robot import Robot

# 1. OTTIMIZZAZIONE WINDOWS: Aumenta la risoluzione del timer a 1ms
ctypes.windll.winmm.timeBeginPeriod(1)

FILENAME = "QUERY_CSV.csv"
COUNTER_FILE = "contatore.csv"
VELOCITY_THRESHOLD = 0.02 # rad/s — modifica questo valore come preferisci

data_queue = queue.Queue()
stop_event = threading.Event()

# ─── Gestione contatore persistente ────────────────────────────────────────────

def load_last_trajectory_id():
    """Legge l'ultimo ID traiettoria dal file contatore. Se non esiste, restituisce 0."""
    if not os.path.exists(COUNTER_FILE):
        return 0
    with open(COUNTER_FILE, mode='r', newline='') as f:
        reader = csv.reader(f)
        rows = list(reader)
        # Salta header, prende l'ultima riga valida
        for row in reversed(rows):
            if row and row[0] != "last_trajectory_id":
                try:
                    return int(row[0])
                except ValueError:
                    continue
    return 0

def save_last_trajectory_id(traj_id: int):
    """Sovrascrive il file contatore con l'ultimo ID usato."""
    with open(COUNTER_FILE, mode='w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["last_trajectory_id"])
        writer.writerow([traj_id])

# ─── Worker CSV principale ──────────────────────────────────────────────────────

def csv_writer_worker(traj_id_ref: dict):
    """
    Scrive le righe nella coda sul CSV principale.
    traj_id_ref è un dict con chiave 'id' per condividere il valore corrente
    tra il thread writer e il thread main (pass-by-reference tramite dict).
    """
    joints = [f"j{i}" for i in range(1, 7)]
    header = ["Timestamp", "trajectory_id"]
    header += [f"{j}_v" for j in joints]
    header += [f"{j}_a" for j in joints]
    header += [f"{j}_t" for j in joints]

    with open(FILENAME, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(header)

        while not stop_event.is_set() or not data_queue.empty():
            try:
                item = data_queue.get(timeout=1.0)
                writer.writerow(item)
                data_queue.task_done()
            except queue.Empty:
                continue

# ─── Main ───────────────────────────────────────────────────────────────────────

def main():
    r = Robot()

    # Carica l'ultimo ID e prepara il riferimento condiviso
    last_id = load_last_trajectory_id()
    current_traj_id = last_id  # ID corrente assegnato alle righe
    traj_id_ref = {"id": current_traj_id}

    # Stato per la logica di cambio traiettoria
    # Una nuova traiettoria inizia quando TUTTE le velocità scendono sotto soglia
    # (robot fermo), e poi riparte → incrementa l'ID alla ripartenza
    robot_was_stopped = False  # True se nell'ultimo ciclo il robot era fermo

    writer_thread = threading.Thread(
        target=csv_writer_worker,
        args=(traj_id_ref,),
        daemon=True
    )
    writer_thread.start()

    print(f"Logging avviato su '{FILENAME}' (timestamp + trajectory_id + 18 col. giunti).")
    print(f"Ultimo trajectory_id caricato da '{COUNTER_FILE}': {last_id}")
    print(f"Threshold velocità: {VELOCITY_THRESHOLD} rad/s")
    print("Premi CTRL+C per fermare.\n")

    try:
        while True:
            try:
                # 1. Lettura dati dal robot
                res_vel = r.get_current_joint_velocities_with_timestamp()
                velocities = res_vel[0]

                # 2. Calcolo magnitudo velocità
                magnitude = math.sqrt(sum(v**2 for v in velocities))

                # 3. Logica di filtraggio e gestione Traiettoria
                if magnitude < VELOCITY_THRESHOLD:
                    # Robot fermo: non scriviamo nulla, segniamo solo lo stato
                    robot_was_stopped = True
                else:
                    # IL ROBOT SI MUOVE: Qui gestiamo l'ID e la scrittura
                    if robot_was_stopped:
                        # Era fermo, ora si muove → nuova traiettoria
                        current_traj_id += 1
                        traj_id_ref["id"] = current_traj_id
                        print(f"  → Nuova traiettoria rilevata: ID = {current_traj_id}")
                    
                    robot_was_stopped = False

                    # Recuperiamo gli altri dati solo se dobbiamo effettivamente scrivere
                    res_ang = r.get_current_joint_angles_with_timestamp()
                    res_trq = r.get_current_joint_torques_with_timestamp()

                    # Creazione della riga e inserimento in coda
                    row = [
                        time.time(),
                        traj_id_ref["id"],
                        *res_vel[0],
                        *res_ang[0],
                        *res_trq[0],
                    ]
                    data_queue.put(row)

            except Exception as e:
                print(f"Errore durante la lettura dati: {e}")

    except KeyboardInterrupt:
        print("\nFermando il logging...")
        stop_event.set()
        writer_thread.join()

        # Salva il contatore aggiornato
        save_last_trajectory_id(traj_id_ref["id"])
        print(f"Contatore aggiornato in '{COUNTER_FILE}': last_trajectory_id = {traj_id_ref['id']}")

        if hasattr(ctypes, 'windll'):
            ctypes.windll.winmm.timeEndPeriod(1)

        print("Log terminato correttamente.")

if __name__ == "__main__":
    main()

[2026-04-01 11:29:47][neurapy_logger][WARNING] : Current client version is not compatiable with the version of the server running on the robot. Some of the functionlities specified in the documentation might not work in the intended way. Please upgrade to the correct version .Client Version : v4.16.4,Server Version : aaed267_v4.14.3 :(robot.py:130)
Logging avviato su 'QUERY_CSV.csv' (timestamp + trajectory_id + 18 col. giunti).
Ultimo trajectory_id caricato da 'contatore.csv': 307
Threshold velocità: 0.02 rad/s
Premi CTRL+C per fermare.

  → Nuova traiettoria rilevata: ID = 308
  → Nuova traiettoria rilevata: ID = 309
  → Nuova traiettoria rilevata: ID = 310
  → Nuova traiettoria rilevata: ID = 311
  → Nuova traiettoria rilevata: ID = 312
  → Nuova traiettoria rilevata: ID = 313
  → Nuova traiettoria rilevata: ID = 314
  → Nuova traiettoria rilevata: ID = 315
  → Nuova traiettoria rilevata: ID = 316
  → Nuova traiettoria rilevata: ID = 317
  → Nuova traiettoria rilevata: ID = 318
  → N

Exception in thread Thread-12:
Traceback (most recent call last):
  File "C:\Users\david\Desktop\KAWASAKI\neurapy\robot.py", line 78, in wrapped_function
    sock.connect(address)
ConnectionAbortedError: [WinError 10053] Connessione interrotta dal software del computer host

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "C:\Users\david\.conda\envs\cobot\lib\threading.py", line 932, in _bootstrap_inner
    self.run()
  File "C:\Users\david\.conda\envs\cobot\lib\site-packages\ipykernel\ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "C:\Users\david\.conda\envs\cobot\lib\threading.py", line 870, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\david\Desktop\KAWASAKI\neurapy\robot.py", line 147, in notify_diagnostics
    diagnostics = self.get_diagnostics()
  File "C:\Users\david\Desktop\KAWASAKI\neurapy\robot.py", line 80, in wrapped_function
    raise ConnectionError("Faile

CONVERTITORE

In [13]:
import os

def converti_csv_per_excel(file_input, file_output):
    # Controlliamo se il file esiste
    if not os.path.exists(file_input):
        print(f"Errore: Il file '{file_input}' non esiste.")
        return

    print(f"Conversione in corso: {file_input} -> {file_output}")
    
    try:
        with open(file_input, 'r', encoding='utf-8') as f_in:
            with open(file_output, 'w', encoding='utf-8') as f_out:
                for linea in f_in:
                    # 1. Sostituisce la virgola (separatore) con punto e virgola
                    # 2. Sostituisce il punto (decimale) con la virgola
                    nuova_linea = linea.replace(',', ';').replace('.', ',')
                    f_out.write(nuova_linea)
        
        print("Conversione completata con successo!")
        
    except Exception as e:
        print(f"Si è verificato un errore: {e}")

# --- CONFIGURAZIONE ---
INPUT = "QUERY_CSV_TOTALE_EXCEL.csv"
OUTPUT = "QUERY_CSV_TOTALE_EXCEL_FINALE.csv"

if __name__ == "__main__":
    converti_csv_per_excel(INPUT, OUTPUT)

Conversione in corso: QUERY_CSV_TOTALE_EXCEL.csv -> QUERY_CSV_TOTALE_EXCEL_FINALE.csv
Conversione completata con successo!


CONVERSIONE A 3 DECIMALI E 5 DECIMALI

In [12]:
import csv

input_file = "QUERY_CSV_TOTALE.csv"
output_file = "QUERY_CSV_TOTALE_EXCEL.csv"

with open(input_file, 'r', newline='') as infile, \
     open(output_file, 'w', newline='') as outfile:

    reader = csv.reader(infile)
    writer = csv.writer(outfile)

    for row in reader:
        if row and row[0].strip().lower() == 'Timestamp':
            writer.writerow(row)
            continue

        formatted_row = []
        for i, value in enumerate(row):
            try:
                num = float(value)
                formatted_row.append(f"{num:.3f}" if i == 0 else f"{num:.5f}")
            except ValueError:
                formatted_row.append(value)

        writer.writerow(formatted_row)